## Overview
A few things to note about `String`s:
- Strings are immutable.
- Can be created either by using a constructor or by assigning a string literal.
- Historically (prior to Java 9), Strings were stored internally as `char[]`, each element containing a 16-bit UTF-16 Unicode character.

The byte representation of a String is expanded below:

In [1]:
import java.nio.charset.Charset;
import java.nio.charset.StandardCharsets;
import java.util.Arrays;

String word = "Hello all!";

System.out.println(Arrays.toString(word.getBytes())); // Sequence of bytes using the default charset: `Charset.defaultCharset()` 
                                                      // UTF-8 by default in Java 18+ via JEP 400
System.out.println(Charset.defaultCharset());

System.out.println(Arrays.toString(word.getBytes(StandardCharsets.UTF_16)));
System.out.println(Arrays.toString(word.getBytes(StandardCharsets.UTF_8)));

[72, 101, 108, 108, 111, 32, 97, 108, 108, 33]
UTF-8
[-2, -1, 0, 72, 0, 101, 0, 108, 0, 108, 0, 111, 0, 32, 0, 97, 0, 108, 0, 108, 0, 33]
[72, 101, 108, 108, 111, 32, 97, 108, 108, 33]


UTF-8 vs UTF-16 and other encoding systems are explained here: [Encoding formats](https://www.youtube.com/watch?v=uTJoJtNYcaQ)

In [2]:
import java.nio.charset.StandardCharsets;

// In Java, String.length() returns the number of 16-bit char code units, NOT bytes or code points!
// U+1F4A9 is outside the BMP (Basic Multilingual Plane), so UTF-16 encodes it as a surrogate pair (2 chars).
System.out.println("Length (code units): " + "💩".length()); // Returns 2

// To get the actual number of Unicode characters (code points):
System.out.println("Code point count: " + "💩".codePointCount(0, "💩".length())); // Returns 1

Length (code units): 2
Code point count: 1


As part of [JEP 254](https://openjdk.org/jeps/254) included in Java 9, Strings are now internally represented differently. Instead of a `char[]` array, Java now uses a `byte[]` and another `byte` variable to store encoding. Most of the time, the characters stored in strings are English characters that can be stored in a single byte. If a string contains characters requiring more than one byte, it encodes them using 2 bytes per character (UTF-16). This provides significant space savings along with reduced GC activity.

## Text Block
Introduced as a preview feature in Java 13 and made permanent in Java 15 ([JEP 378](https://openjdk.org/jeps/378)), text blocks let us construct multiline strings in a less verbose fashion.

In [3]:
System.out.println("""
                       <person>
                           <firstName>Bob</firstName>
                           <lastName>Jones</lastName>
                       </person>
                       """);

// Note that we don't have to escape " characters inside
System.out.println("""
                   {
                       "name": "John",
                       "age": 54
                   }
                   """);

<person>
    <firstName>Bob</firstName>
    <lastName>Jones</lastName>
</person>

{
    "name": "John",
    "age": 54
}



Notice that the indentation before the "\<person>" opening and closing tags are gone - part of reason why this is called text block and not multiline string literals.

In [4]:
// Literal needs to have line break after """
System.out.println("""Some long lines of text below.
Starting at the very beginning of the text.
    We have some more indented text.
Back to no indentation.
                   """);

CompilationException: [snippet: [// Literal needs to have line break after """
System.out.println("""Some long lines of text below.
Starting at the very beginning of the text.
    We have some more indented text.
Back to no indentation.
                   """);], status: REJECTED]

In [ ]:
// Equivalent to "                   Some long lines of text below.\nStarting at the very beginning of the text.\n    We have some more indented text.\nBack to no indentation.\n"
System.out.println("""
                   Some long lines of text below.
Starting at the very beginning of the text.
    We have some more indented text.
Back to no indentation.
                   """);

// Equivalent to "                   Some long lines of text below.\nStarting at the very beginning of the text.\n    We have some more indented text.\nBack to no indentation."                   
// (Closing delimiter on the same line as text omits the trailing newline)
System.out.println("""
                   Some long lines of text below.
Starting at the very beginning of the text.
    We have some more indented text.
Back to no indentation.""");
                   
// Equivalent to "                   Some long lines of text below.\nStarting at the very beginning of the text.\n    We have some more indented text.\nBack to no indentation."                   
// Trailing whitespace is removed
System.out.println("""
                   Some long lines of text below.
Starting at the very beginning of the text.
    We have some more indented text.
Back to no indentation.    """);

To prevent automatic insertion of a newline at the end of a line, we can use the escape character `\<line-terminator>`:

In [5]:
String longText = """
     Warning! An error has occurred.\
      Please go to the home page. """;
System.out.println(longText);

Warning! An error has occurred. Please go to the home page.


To preserve spaces at the end of a line, use `\s` since incidental trailing whitespace normally gets removed:

In [6]:
String colours = """
    red  \s
    green\s
    blue \s
    """;
System.out.println(colours);

// Without \s, the string would be "red\ngreen\nblue\n"

red   
green 
blue  



## String Pool
Whenever we use string literals to create a string variable, these literals are stored in an area of the heap called the **String Pool** (or String Constant Pool). If a literal with the same content already exists, the JVM reuses the reference rather than creating a new object.

In [7]:
String a = "Java";    // "Java" added to String Pool
String b = "Java";    // b refers to the same instance in String Pool

String c = new String("Java");    // new object created

if(a == b)
    System.out.println("a and b are same");    // same
else
    System.out.println("a and b are diff");

if(a == c)
    System.out.println("a and c are same");
else
    System.out.println("a and c are diff");    // diff

if(a.equals(c))
    System.out.println("a and c have same content"); // same content
else
    System.out.println("a and c have diff content");

// intern() returns the canonical instance from the String Pool:
if (a == c.intern())
    System.out.println("c.intern() refers to the pooled instance 'a'"); // true

a and b are same
a and c are diff
a and c have same content
c.intern() refers to the pooled instance 'a'


## StringBuilder and StringBuffer
Below is the hierarchy `String` is part of:
```
CharSequence(I)
        +-- String(C)
        +-- StringBuilder(C)
        +-- StringBuffer(C)
```

`StringBuilder` and `StringBuffer` are two mutable companion classes to `String`. Both provide methods such as `append()`, `insert()`, `delete()`, and `reverse()` to modify contents in-place.

`StringBuffer` is the thread-safe version of `StringBuilder`. If multiple threads need to access `StringBuilder`, it is better to use `StringBuffer`.

In [13]:
StringBuilder sb = new StringBuilder("Hello");
sb.append(" ").append("World");
sb.insert(5, ",");
System.out.println("Appended: " + sb);

sb.reverse();
System.out.println("Reversed: " + sb);
sb.reverse(); // reverse back

sb.delete(5, 6); // remove comma
System.out.println("After delete: " + sb.toString());

Appended: Hello, World
Reversed: dlroW ,olleH
After delete: Hello World


## Modern String APIs
**Java 8**

In [14]:
// Character stream
"String".chars().forEach(System.out::println);

83
116
114
105
110
103


**Java 11**

In [15]:
String text = "   Hello World   ";
// Strip is Unicode whitespace character aware (trim is not)
System.out.println(text.strip());
System.out.println(text.repeat(2));
// Checks if String only has whitespaces
System.out.println(text.isBlank());

String multiLine = "A big sentence\ndivided across\nmultiple lines";
// generates a stream of lines
multiLine.lines().forEach(line -> System.out.print(line + " "));

Hello World
   Hello World      Hello World   
false
A big sentence divided across multiple lines 

**Java 12**

In [16]:
String text = "Hello World";
// Adds n spaces to the start
System.out.println(text.indent(4));
// Applies a function to String
System.out.println(text.transform(str -> new StringBuilder(str).reverse().toString()).toString());

    Hello World

dlroW olleH


**Java 15**

In [17]:
// formatted(Object... args): Instance method equivalent to String.format(...)
String formatted = "Hello %s, your balance is $%.2f".formatted("Alice", 149.50);
System.out.println(formatted);

// stripIndent(): Programmatically strips incidental indentation from multiline strings
String raw = "    line 1\n    line 2\n";
System.out.println(raw.stripIndent());

// translateEscapes(): Translates escape sequences (e.g. \t, \n) in a raw string
String escaped = "Hello\tWorld\n";
System.out.print(escaped.translateEscapes());

Hello Alice, your balance is $149.50
    line 1
    line 2

Hello	World
